<a href="https://colab.research.google.com/github/irajamuller/grover_adiabatico_experiments/blob/main/UA1_UA2_Algoritmo_de_Grover_Adiab%C3%A1tico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install dwave-ocean-sdk pyqubo --quiet

In [26]:
dimod.__version__

'0.12.21'

In [3]:
import dimod
import dwave.samplers as samplers
import numpy as np
from collections import Counter
import time

In [29]:
def construir_oracle_qubo(estado_alvo: str) -> dimod.BinaryQuadraticModel:
    """
    Constrói o BQM (Binary Quadratic Model) que representa o oráculo de
    Grover para o estado-alvo dado.
    Args:
        estado_alvo: string binária, ex: "11101"

    Returns:
        dimod.BinaryQuadraticModel no espaço BINARY
    """
    n = len(estado_alvo)
    linear = {}
    offset = 0.0

    for i, bit in enumerate(estado_alvo):
        wi = int(bit)
        # Equivalente a minimizar Σ (qᵢ - wᵢ)² expandido para QUBO binário
        linear[i] = 1.0 - 2.0 * wi   # coeficiente de qᵢ
        offset += wi                  # constante (deslocamento de energia)

    bqm = dimod.BinaryQuadraticModel(linear, {}, offset, dimod.BINARY)

    # Para trace
    print(f"\nEquação Hf: minimize H(q) = " + " + ".join([f"({linear[i]})*q_{i}" for i in range(n)]) + f" + {offset}")
    print(f"H0 (Teórico): -Σ σ_x_i (Superposição uniforme de todos os 2^{n} estados)")
    return bqm

def executar_grover_dwave(
    estado_alvo: str,
    num_reads: int = 2000
) -> tuple:
    """
    Executa o oráculo de Grover como problema QUBO no amostrador D-Wave
    Ocean (Simulated Annealing via dwave-samplers).

    Args:
        estado_alvo:  string binária alvo
        num_leituras: número de amostras independentes
        verbose:      imprimir tabela de resultados

    Returns:
        (response, lista_resultados_ordenados)
    """
    n = len(estado_alvo)
    bqm = construir_oracle_qubo(estado_alvo)

    sampler = samplers.SimulatedAnnealingSampler()

    t0 = time.time()
    response = sampler.sample(
        bqm,
        num_reads=num_reads,
        num_sweeps=2000,                 # "comprimento" do annealing
        beta_range=(0.05, 15.0),         # temperatura: quente → frio
        beta_schedule_type="geometric",  # resfriamento geométrico (padrão D-Wave)
        seed=42,
    )
    tempo_execucao = time.time() - t0

    total_leituras = num_reads

    resultados = []
    freq_estados = Counter()

    for amostra, energia, n_occ in response.data(["sample", "energy", "num_occurrences"]):
        estado_str = "".join(str(amostra[i]) for i in range(n))
        freq_estados[estado_str] += int(n_occ)

    for estado_str, contagem in sorted(freq_estados.items(),
                                        key=lambda x: -x[1]):
        energia = sum(
            bqm.linear[i] * int(estado_str[i]) for i in range(n)
        ) + bqm.offset
        prob = contagem / total_leituras
        resultados.append((estado_str, energia, contagem, prob))

    resultados.sort(key=lambda x: x[1])   # ordena por energia crescente
    return response, resultados, tempo_execucao


def run(estado_alvo: str) -> None:
    n = len(estado_alvo)
    N = 2 ** n
    num_reads=1024
    sep = "═" * 100

    print(f"\n{'─'*100}")
    print(f"ESTADO-ALVO {estado_alvo}")
    print(f"{'─'*100}")

    response, resultados, tempo_execucao = executar_grover_dwave(estado_alvo, num_reads=num_reads)

    print(f"\n{'─'*100}")
    print(f"RESUMO FINAL")
    print(f"{'─'*100}")

    resultado_alvo = next((r for r in resultados if r[0] == estado_alvo), None)
    if resultado_alvo:
        _, e_alvo, c_alvo, p_alvo = resultado_alvo
        print(f"Energia QUBO  : {e_alvo:.4f}  (mínimo global)")
        print(f"Taxa de acerto: {p_alvo:.2%}  ({c_alvo}/{num_reads} leituras)")
        print(f"Tempo de execução: {tempo_execucao:.2f} ")
    print(f"\n{sep}\n")

In [33]:
if __name__ == '__main__':
  estados_alvo = ["101", "11101", "1010100101", "10101001011010100101"]
  for alvo in estados_alvo:
    run(estado_alvo=alvo)


────────────────────────────────────────────────────────────────────────────────────────────────────
ESTADO-ALVO 101
────────────────────────────────────────────────────────────────────────────────────────────────────

Equação Hf: minimize H(q) = (-1.0)*q_0 + (1.0)*q_1 + (-1.0)*q_2 + 2.0
H0 (Teórico): -Σ σ_x_i (Superposição uniforme de todos os 2^3 estados)

────────────────────────────────────────────────────────────────────────────────────────────────────
RESUMO FINAL
────────────────────────────────────────────────────────────────────────────────────────────────────
Energia QUBO  : 0.0000  (mínimo global)
Taxa de acerto: 100.00%  (1024/1024 leituras)
Tempo de execução: 0.14 

════════════════════════════════════════════════════════════════════════════════════════════════════


────────────────────────────────────────────────────────────────────────────────────────────────────
ESTADO-ALVO 11101
─────────────────────────────────────────────────────────────────────────────────────────